In [ ]:
# Persistance helps in saving and maintaining the state and checkpoints and helps in various things like :
# 1: Human in the Loop - as it posesses the power to resume the processes from any checkpoint , so it make it easier to resume the processes from the last executed step 
# 2 : Fault tolerance - similarly as if the code crashes at any checkpoint we dont losses the entire progress made till that point 
# 3 : Memory : As it has the options of both the Short term and long term DB options to store the state values , it doesnt losses the progress and  the user can resume the process anytime (Ps : the short term is a ram storage hence suitable for the testing and debugging purpose only )
# 4 : Time Travel : we can go to any checkpoint or the node by using its checkpoint id , and can start invoking the process again from that step , helps in the debugging and analysing the code better 

In [ ]:

from dotenv import load_dotenv
from typing import TypedDict , Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph , START, END 
from langgraph.checkpoint.memory import InMemorySaver# A checkpointer for the memory of the state only for the short term RAM Based memory save option 
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage , HumanMessage , AIMessage
load_dotenv()

True

In [85]:
class state_Schema(TypedDict):
    topic : str
    joke : Annotated[list[BaseMessage] , add_messages]
    evaluation : Annotated[list[BaseMessage] , add_messages]

In [86]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [87]:
graph = StateGraph(state_Schema)

In [88]:
checkpointer = InMemorySaver()

In [89]:
def joke_node(state:state_Schema):
    inpt = state['topic']

    res = model.invoke(f"write a joke on the topic :{inpt} , avoid conversation with the user , start the joke directly.")

    return {'joke':[res.content]}

In [90]:
def evaluation_node(state : state_Schema):

    inpt = state['joke']

    res = model.invoke(f"evaluate the joke on various factors  in 50 words: {inpt}")

    return {'evaluation':[res.content]}

In [91]:
graph.add_node("joke_node", joke_node)
graph.add_node("evaluation_node" , evaluation_node)

In [92]:
graph.add_edge(START,"joke_node")
graph.add_edge("joke_node","evaluation_node")
graph.add_edge("evaluation_node",END)

In [93]:
workflow = graph.compile(checkpointer=checkpointer)
thread_id = "2"
config = {"configurable":{"thread_id":thread_id}}


In [94]:
inpt = input("enter the topic of joke")
intial_state = {'topic':inpt}
res = workflow.invoke(intial_state , config=config)
print(res)

{'topic': 'pizza', 'joke': [HumanMessage(content='Why did the pizza chef get fired? He kept making cheesy jokes.', additional_kwargs={}, response_metadata={}, id='bfc7a9d6-6c1c-47bf-8a92-3d005d596c02')], 'evaluation': [HumanMessage(content='This is a classic "dad joke," relying on a simple, effective pun. The wordplay on "cheesy" (pizza ingredient and bad jokes) is clear and easily understood. It elicits a groan or a mild chuckle rather than a big laugh, making it harmless and highly accessible, though lacking in originality or sophisticated wit.', additional_kwargs={}, response_metadata={}, id='c4f13cd7-f58e-4119-8ea9-1f346e88590a')]}


In [ ]:
# to get the current executable state value 
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': [HumanMessage(content='Why did the pizza chef get fired? He kept making cheesy jokes.', additional_kwargs={}, response_metadata={}, id='bfc7a9d6-6c1c-47bf-8a92-3d005d596c02')], 'evaluation': [HumanMessage(content='This is a classic "dad joke," relying on a simple, effective pun. The wordplay on "cheesy" (pizza ingredient and bad jokes) is clear and easily understood. It elicits a groan or a mild chuckle rather than a big laugh, making it harmless and highly accessible, though lacking in originality or sophisticated wit.', additional_kwargs={}, response_metadata={}, id='c4f13cd7-f58e-4119-8ea9-1f346e88590a')]}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e5b-8d87-62ad-8002-721f0a465192'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-27T07:05:44.597162+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e5b-5d

In [96]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': [HumanMessage(content='Why did the pizza chef get fired? He kept making cheesy jokes.', additional_kwargs={}, response_metadata={}, id='bfc7a9d6-6c1c-47bf-8a92-3d005d596c02')], 'evaluation': [HumanMessage(content='This is a classic "dad joke," relying on a simple, effective pun. The wordplay on "cheesy" (pizza ingredient and bad jokes) is clear and easily understood. It elicits a groan or a mild chuckle rather than a big laugh, making it harmless and highly accessible, though lacking in originality or sophisticated wit.', additional_kwargs={}, response_metadata={}, id='c4f13cd7-f58e-4119-8ea9-1f346e88590a')]}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e5b-8d87-62ad-8002-721f0a465192'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-27T07:05:44.597162+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e5b-5

In [98]:

workflow.update_state({"configurable":{"thread_id":thread_id , "checkpoint_id":"1f1a1e5b-1b75-6864-8000-f63a2b85d037" , "checkpoint_ns":""}},{'topic':"watermelon"})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a1e92-03dc-6ce3-8001-161591d57bf7'}}

In [100]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'watermelon', 'joke': [], 'evaluation': []}, next=('joke_node',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e92-03dc-6ce3-8001-161591d57bf7'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-27T07:30:06.556899+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e5b-1b75-6864-8000-f63a2b85d037'}}, tasks=(PregelTask(id='4267ba75-6568-2524-c143-6367852b93c9', name='joke_node', path=('__pregel_pull', 'joke_node'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [HumanMessage(content='Why did the pizza chef get fired? He kept making cheesy jokes.', additional_kwargs={}, response_metadata={}, id='bfc7a9d6-6c1c-47bf-8a92-3d005d596c02')], 'evaluation': [HumanMessage(content='This is a classic "dad joke," relying on a simple, effective pun. The wordplay on "cheesy" (piz

In [ ]:
# here we copied the id of the checkpoint we updated in the previous cell , and invoked the workflow for that , and we passed the None as WE ARE RESUMING THE WORKFLOW AND DONT WANTS TO AGAIN START FROM THE START NODE .
workflow.invoke(None , {"configurable":{"thread_id":thread_id , "checkpoint_id":"1f1a1e92-03dc-6ce3-8001-161591d57bf7","checkpoint_ns":""}})

{'topic': 'watermelon',
 'joke': [HumanMessage(content='Why was the watermelon always invited to parties?\nBecause it was one in a melon!', additional_kwargs={}, response_metadata={}, id='ad5b353d-9dd4-4f68-97fc-91484122b6d2')],
 'evaluation': [HumanMessage(content='This is a classic, charming pun. The "one in a melon" wordplay is the joke\'s strong point, making it clever and accessible. While not groundbreakingly original, it delivers a predictable, lighthearted chuckle or groan, typical of a dad joke. It\'s effective for a quick, harmless laugh and widely understood.', additional_kwargs={}, response_metadata={}, id='8de748e8-97db-4b47-adc6-8d0ebd4b6ea5')]}

In [ ]:
# to get the state history 
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'watermelon', 'joke': [HumanMessage(content='Why was the watermelon always invited to parties?\nBecause it was one in a melon!', additional_kwargs={}, response_metadata={}, id='ad5b353d-9dd4-4f68-97fc-91484122b6d2')], 'evaluation': [HumanMessage(content='This is a classic, charming pun. The "one in a melon" wordplay is the joke\'s strong point, making it clever and accessible. While not groundbreakingly original, it delivers a predictable, lighthearted chuckle or groan, typical of a dad joke. It\'s effective for a quick, harmless laugh and widely understood.', additional_kwargs={}, response_metadata={}, id='8de748e8-97db-4b47-adc6-8d0ebd4b6ea5')]}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a1e96-d8e3-6589-8003-4de7c610ba5b'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-27T07:32:16.268429+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'check